In [1]:
# CELL 1 — Imports & config
import pandas as pd
import os

BASE_PATH   = r"C:\Users\Lenovo\Downloads\nyc_taxi_project"
SILVER_PATH = os.path.join(BASE_PATH, "silver", "yellow_trips_silver.parquet")
GOLD_DIR    = os.path.join(BASE_PATH, "gold")

os.makedirs(GOLD_DIR, exist_ok=True)

In [2]:
# CELL 2 — Load Silver
df = pd.read_parquet(SILVER_PATH)
print(f"Silver records loaded: {len(df):,}")

Silver records loaded: 2,883,081


In [3]:
# CELL 3 — Gold Table 1: Hourly stats
gold_hourly = (df
    .groupby("pickup_hour")
    .agg(
        total_trips       = ("pickup_hour", "count"),
        avg_fare          = ("fare_amount", "mean"),
        avg_duration_mins = ("trip_duration_minutes", "mean"),
        avg_tip_pct       = ("tip_percentage", "mean"),
        total_revenue     = ("total_amount", "sum")
    )
    .round(2)
    .reset_index()
    .sort_values("pickup_hour")
)
print("Hourly Stats:")
display(gold_hourly)

Hourly Stats:


,pickup_hour,total_trips,avg_fare,avg_duration_mins,avg_tip_pct,total_revenue
0,0,79531,19.79,15.67,20.42,2295238.95
1,1,55528,17.82,14.90,20.96,1460272.16
2,2,38701,16.70,14.66,20.72,959300.15
3,3,25028,17.72,14.59,20.26,650732.50
4,4,15806,22.18,15.81,17.99,496456.57
5,5,16128,26.43,15.70,16.57,590591.94
6,6,39923,22.13,15.16,18.12,1220036.55
7,7,79518,18.90,15.14,37.66,2130970.69
8,8,107864,17.42,15.70,20.02,2709190.80
9,9,122596,17.60,15.34,19.91,3127041.51


In [4]:
# CELL 4 — Gold Table 2: Daily revenue
gold_daily = (df
    .groupby("pickup_date")
    .agg(
        total_trips        = ("pickup_date", "count"),
        daily_revenue      = ("total_amount", "sum"),
        avg_fare           = ("fare_amount", "mean"),
        avg_distance_miles = ("trip_distance_miles", "mean"),
        avg_duration_mins  = ("trip_duration_minutes", "mean")
    )
    .round(2)
    .reset_index()
    .sort_values("pickup_date")
)
print("Daily Revenue:")
display(gold_daily)

Daily Revenue:


,pickup_date,total_trips,daily_revenue,avg_fare,avg_distance_miles,avg_duration_mins
0,2008-12-31,1,80.55,70.00,17.76,927.48
1,2022-10-25,4,159.20,37.25,2.45,6.87
2,2022-12-31,25,658.23,16.63,3.31,12.21
3,2023-01-01,69968,2156065.54,21.94,4.38,17.54
4,2023-01-02,61858,1946192.83,22.31,4.64,16.41
5,2023-01-03,80528,2371351.96,20.25,3.94,16.60
6,2023-01-04,89510,2521817.48,19.17,3.60,16.32
7,2023-01-05,95164,2605814.42,18.51,3.43,16.09
8,2023-01-06,96600,2582506.63,17.94,3.26,15.55
9,2023-01-07,99147,2546027.64,17.72,3.28,14.95


In [5]:
# CELL 5 — Gold Table 3: Payment breakdown
gold_payment = (df
    .groupby("payment_type_label")
    .agg(
        total_trips   = ("payment_type_label", "count"),
        avg_tip_pct   = ("tip_percentage", "mean"),
        total_revenue = ("total_amount", "sum")
    )
    .round(2)
    .reset_index()
    .sort_values("total_trips", ascending=False)
)
print("Payment Breakdown:")
display(gold_payment)

Payment Breakdown:


,payment_type_label,total_trips,avg_tip_pct,total_revenue
1,Credit Card,2350156,25.97,66246302.91
0,Cash,507711,0.00,12008731.60
2,Dispute,16220,0.14,389905.56
3,No Charge,8994,0.02,202850.11


In [6]:
# CELL 6 — Gold Table 4: Top 10 busiest zones
gold_zones = (df
    .groupby("pickup_location_id")
    .agg(
        total_pickups = ("pickup_location_id", "count"),
        avg_fare      = ("fare_amount", "mean"),
        avg_distance  = ("trip_distance_miles", "mean")
    )
    .round(2)
    .reset_index()
    .sort_values("total_pickups", ascending=False)
    .head(10)
)
print("Top 10 Busiest Zones:")
display(gold_zones)

Top 10 Busiest Zones:


,pickup_location_id,total_pickups,avg_fare,avg_distance
122,132,151975,60.72,15.51
225,237,141096,12.35,1.80
224,236,130870,13.13,1.97
151,161,129081,15.26,2.39
175,186,104793,15.55,2.32
152,162,100785,14.89,2.36
132,142,94899,13.60,2.19
218,230,94196,17.38,2.99
128,138,86720,41.49,9.68
160,170,83972,14.83,2.32


In [7]:
# CELL 7 — Save all Gold tables as CSV
gold_tables = {
    "hourly_stats"      : gold_hourly,
    "daily_revenue"     : gold_daily,
    "payment_breakdown" : gold_payment,
    "top_zones"         : gold_zones,
}

for name, table in gold_tables.items():
    path = os.path.join(GOLD_DIR, f"{name}.csv")
    table.to_csv(path, index=False)
    print(f"Saved: {path}")

print("\nPipeline complete! Bronze -> Silver -> Gold")
print(f"Your CSVs are in: {GOLD_DIR}")

Saved: C:\Users\Lenovo\Downloads\nyc_taxi_project\gold\hourly_stats.csv
Saved: C:\Users\Lenovo\Downloads\nyc_taxi_project\gold\daily_revenue.csv
Saved: C:\Users\Lenovo\Downloads\nyc_taxi_project\gold\payment_breakdown.csv
Saved: C:\Users\Lenovo\Downloads\nyc_taxi_project\gold\top_zones.csv

Pipeline complete! Bronze -> Silver -> Gold
Your CSVs are in: C:\Users\Lenovo\Downloads\nyc_taxi_project\gold
